In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. 데이터 불러오기
df = pd.read_csv('final_4_exercises_dataset.csv')

# 2. 운동 명칭 단순화 (팀 프로젝트 발표 시 보기 좋게!)
mapping = {
    '스텝 백워드 다이나믹 런지': 'Lunge',
    '푸시업': 'Pushup',
    '바벨 스쿼트': 'Squat',
    '플랭크': 'Plank'
}
df['exercise'] = df['exercise'].map(mapping)

# 3. 좌표 데이터 중 비어있는 값(0) 처리 및 정규화 준비
# (관절 좌표 컬럼들만 선택)
feature_cols = [col for col in df.columns if '_x' in col or '_y' in col]
X = df[feature_cols].fillna(0) # 비어있는 좌표는 0으로 채움
y = df['exercise']

# 4. 학습 데이터와 테스트 데이터 나누기 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ 데이터 준비 완료!")
print(f"학습용 데이터: {len(X_train)}행")
print(f"검증용 데이터: {len(X_test)}행")
print("\n📊 학습할 종목별 비율:\n", y_train.value_counts(normalize=True))

✅ 데이터 준비 완료!
학습용 데이터: 82201행
검증용 데이터: 20551행

📊 학습할 종목별 비율:
 exercise
Lunge     0.452805
Pushup    0.259109
Squat     0.223623
Plank     0.064464
Name: proportion, dtype: float64


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib # 모델 저장용
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 모델 생성 (데이터 불균형을 고려해 class_weight='balanced' 설정)
print("🏋️ AI 모델 학습 중... 잠시만 기다려 주세요.")
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# 2. 모델 학습
model.fit(X_train, y_train)

# 3. 예측 및 평가
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("-" * 30)
print(f"✅ 학습 완료! 모델 정확도: {accuracy*100:.2f}%")
print("-" * 30)

# 4. 상세 결과 출력 (종목별로 얼마나 잘 맞히는지)
print("📊 종목별 학습 리포트:")
print(classification_report(y_test, y_pred))

# 5. 모델 저장 (나중에 웹이나 앱에서 쓰기 위해 저장합니다)
joblib.dump(model, 'exercise_model.pkl')
print("💾 모델 저장 완료: 'exercise_model.pkl'")

🏋️ AI 모델 학습 중... 잠시만 기다려 주세요.
------------------------------
✅ 학습 완료! 모델 정확도: 99.88%
------------------------------
📊 종목별 학습 리포트:
              precision    recall  f1-score   support

       Lunge       1.00      1.00      1.00      9305
       Plank       1.00      0.99      1.00      1325
      Pushup       1.00      1.00      1.00      5325
       Squat       1.00      1.00      1.00      4596

    accuracy                           1.00     20551
   macro avg       1.00      1.00      1.00     20551
weighted avg       1.00      1.00      1.00     20551

💾 모델 저장 완료: 'exercise_model.pkl'


In [1]:
import mediapipe as mp

In [2]:
import sys
print(sys.executable)

C:\Users\smhrd1\anaconda3\envs\gym\python.exe


In [3]:
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

print("정상 작동 👍")


정상 작동 👍


In [1]:
import cv2
import joblib
import numpy as np
import mediapipe as mp
import pandas as pd 

# 솔루션 초기화
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# 모델 로드 (경로 확인 필수)
try:
    model = joblib.load('exercise_model.pkl')
except FileNotFoundError:
    print("모델 파일을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()

pose_tracker = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# 사용할 관절 상수 리스트 (이름 대신 상수를 직접 써서 오타 방지)
target_joints = [
    mp_pose.PoseLandmark.NOSE,
    mp_pose.PoseLandmark.LEFT_SHOULDER, mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW, mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.LEFT_HIP, mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE, mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE, mp_pose.PoseLandmark.RIGHT_ANKLE
]

cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, image = cap.read()
    if not success: break

    image = cv2.flip(image, 1)
    results = pose_tracker.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        
        # 랜드마크 추출
        try:
            input_row = []
            for joint in target_joints:
                # 학습 데이터와 동일한 스케일링인지 확인 (현재 *1000 적용됨)
                input_row.extend([landmarks[joint].x * 1000, landmarks[joint].y * 1000])

            # 리스트를 numpy array로 변환 (속도 및 호환성)
            # input_data = np.array([input_row])
            # 컬럼명 생성 (학습 때랑 동일해야 함)
            columns = []
            for joint in target_joints:
                name = joint.name.lower()
                columns.extend([f'{name}_x', f'{name}_y'])

            input_data = pd.DataFrame([input_row], columns=columns)

            # 예측 및 확률 계산
            prediction = model.predict(input_data)[0]
            prob = np.max(model.predict_proba(input_data))

            # 화면 출력
            text = f'{prediction} ({prob*100:.1f}%)'
            cv2.putText(image, text, (30, 70), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            
            # 골격 그리기
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
            
        except Exception as e:
            # 에러 내용을 콘솔에 출력하여 원인 파악
            print(f"Prediction Error: {e}")

    cv2.imshow('3.13 AI Trainer Test', image)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()